In [46]:
from deck import full_euchre_deck
from dealer import Dealer
# from numba import njit
from n_branches import array_set_difference
from n_play_round import round1, next_round
from tree_search import definitive_winner, find_best_opener, n_trick_sim, trick_played
import numpy as np


In [69]:
game = Dealer(deck=full_euchre_deck ,players=4)
# game.stack_deck(stack_cards=dealer_pick, player=4)
game.deal_cards()
hands_test = np.array([game.hand1, game.hand2, game.hand3, game.hand4])
hands_test

array([[[  0,  -9],
        [-14,   0],
        [-10,   0],
        [  0, -10],
        [ 13,   0]],

       [[  0, -14],
        [-11,   0],
        [  0,  90],
        [  9,   0],
        [ 14,   0]],

       [[-13,   0],
        [-12,   0],
        [ 12,   0],
        [  0, 130],
        [ 11,   0]],

       [[  0, -12],
        [ -9,   0],
        [  0, 120],
        [  0, 140],
        [ 10,   0]]])

In [70]:
# hands_test = np.array([[[-10,   0],
#         [-11,   0],
#         [-14,   0],
#         [-12,   0],
#         [  0, -10]],

#        [[  0, -12],
#         [  0, -13],
#         [  0, -14],
#         [-13,   0],
#         [  0, 100]],

#        [[ 14,   0],
#         [  0, 135],
#         [ 10,   0],
#         [  0,  90],
#         [  0,  -9]],

#        [[ 12,   0],
#         [  0, 140],
#         [ 11,   0],
#         [  9,   0],
#         [  0, 110]]])

In [71]:
# starting_lead = 0
# previous_winners = np.array([])

In [72]:
best_opener = find_best_opener(lead=0, hands=hands_test, tricks=5, previous_winners=np.array([]), sim_func=n_trick_sim, verbose=True)

[0.29539749 0.39420935 0.4853114  0.32063197 0.35455278]


In [73]:
# r2_leads, r2_hands = round1(
#     hands_dealt=hands_test, chosen_card=best_opener, leader=starting_lead
# )

In [74]:
# r2_hands.shape[0]

In [ ]:
# got this working, now I just need to build out the the cases for 4, 3, 2, and 1 tricks left.

def player_best_prune(
    player: int,
    first_response: bool,
    best_opener: int = 0,
    starting_lead: int = 0,
    tricks: int = 5,
    given_hands: np.ndarray = np.array([], dtype=np.int64),
    given_leads: np.ndarray = np.array([], dtype=np.int64),
    previous_winners: np.ndarray = np.array([], dtype=np.int64)

):

    if first_response:
        given_leads, given_hands = round1(
            hands_dealt=hands_test, chosen_card=best_opener, leader=starting_lead
        )

    # else:
    #     given_hands = given_hands
    #     given_leads = given_leads

    player = player
    responder = (starting_lead + player) % 4

    possible_plays_responder = np.unique(given_hands[:, responder], axis=0)

    prune_masks = np.zeros(
        shape=(possible_plays_responder.shape[0], given_hands.shape[0]), dtype=np.bool
    )
    best_pruned_branch_res = np.zeros(
        possible_plays_responder.shape[0], dtype=np.float64
    )

    pruned_idx = 0

    for target in possible_plays_responder:
        mask = np.all(given_hands[:, responder] == target, axis=(1, 2))
        filtered_hands = given_hands[
            mask
        ]  # need to save these at the end based on what produces the best value
        filtered_leads = given_leads[mask]

        r3_leads, r3_hands, r3_score = next_round(
            current_hands=filtered_hands,
            leads=filtered_leads,
            game_round=2,
            game_score=filtered_leads.reshape(-1, 1),
        )
        r4_leads, r4_hands, r4_score = next_round(
            current_hands=r3_hands, leads=r3_leads, game_round=3, game_score=r3_score
        )
        r5_leads, r5_hands, r5_score = next_round(
            current_hands=r4_hands, leads=r4_leads, game_round=4, game_score=r4_score
        )
        r6_leads, r6_hands, r6_score = next_round(
            current_hands=r5_hands, leads=r5_leads, game_round=5, game_score=r5_score
        )
        results = r6_score.reshape(r6_score.shape[0], 5)

        # Calculate meta results
        meta_results = np.zeros(results.shape[0], dtype=np.int64)

        for i in range(len(results)):
            # Count wins for odd-numbered players (team 1)
            total_odd_wins = np.sum(results[i] % 2)

            # Add previous winners to the count
            if len(previous_winners) > 0:
                total_odd_wins += np.sum(previous_winners % 2)

            # Team wins if they get 3+ tricks out of 5 total
            if total_odd_wins >= 3:
                score = 0
            else:
                score = 1

            meta_results[i] = score

        prune_masks[pruned_idx] = mask
        best_pruned_branch_res[pruned_idx] = np.mean(meta_results)
        pruned_idx += 1
        # print(filtered_hands.shape)
        # print(filtered_hands)

    # final_pruned_branch = best_pruned_branch_hands[np.argmax(best_pruned_branch_res)].shape
    if responder % 2 == 0:
        best_prune = prune_masks[np.argmax(best_pruned_branch_res)]
    else: 
        best_prune = prune_masks[np.argmin(best_pruned_branch_res)]

    final_prune = given_hands[best_prune]
    leads_prune = given_leads[best_prune]

    return final_prune, leads_prune


In [76]:
next_best_1, next_best_1_leads = player_best_prune(player=1, first_response=True, best_opener=best_opener, starting_lead=0)

In [77]:
next_best_2, next_best_2_leads = player_best_prune(player=2, first_response=False, best_opener=best_opener, starting_lead=0, given_hands=next_best_1, given_leads=next_best_1_leads)

In [78]:
next_best_3, next_best_3_leads = player_best_prune(player=3, first_response=False, best_opener=best_opener, starting_lead=0, given_hands=next_best_2, given_leads=next_best_2_leads)

In [83]:
hands_test

array([[[  0,  -9],
        [-14,   0],
        [-10,   0],
        [  0, -10],
        [ 13,   0]],

       [[  0, -14],
        [-11,   0],
        [  0,  90],
        [  9,   0],
        [ 14,   0]],

       [[-13,   0],
        [-12,   0],
        [ 12,   0],
        [  0, 130],
        [ 11,   0]],

       [[  0, -12],
        [ -9,   0],
        [  0, 120],
        [  0, 140],
        [ 10,   0]]])

In [82]:
trick_played(hands_test, next_best_3[0])

array([[-10,   0],
       [-11,   0],
       [-12,   0],
       [ -9,   0]])

In [79]:
find_best_opener(
    hands=next_best_3[0],
    lead=next_best_3_leads[0],
    tricks=4,
    previous_winners=np.array([next_best_3_leads[0]], dtype=np.int64),
    sim_func=n_trick_sim,
    verbose=True,
)

[0.13291139 0.18518519 0.19047619 0.18518519]


np.int64(2)